# Create Bulk Drop Statements

This notebook automates the creation of SQL `DROP` statements to remove database objects (`TABLE` or `VIEW`) like tables or views safely.

## Overview
- **Problem Statement**
  When migrating from the Enterprise account to Galaxy in Starburst, ran into an issue with the permissions of the table ownership for dbt generated tables. Therefore, 
- **Purpose:**  
  Automate the generation of SQL `DROP` statements to reduce manual effort and minimize errors.

- **Process:**  
  1. **Input Collection:** Get a list of database objects.  
  2. **Statement Generation:** Automatically create `DROP` statements (e.g., `DROP TABLE IF EXISTS table_name;`).  
  3. **Review:** Display the generated statements that can be run directly in the database console .  

# Imports

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
from trino.dbapi import connect
from trino.auth import OAuth2Authentication

conn_starburst = connect(
    host = 'starburst',
    port = 8443,
    user = '',
    auth = OAuth2Authentication(),
    http_scheme ='https'
)

Validate the system query: 

~~~sql
SELECT 
    'DROP TABLE IF EXISTS ' || table_schem || '.' || table_name || ';' AS table_drop_statement
FROM
    system.jdbc.tables
WHERE 
    True 
    AND table_type = 'TABLE'
    AND table_cat = 'prod'
~~~

Alternatively, `TABLE` can be replaced with the `VIEW`

In [3]:
existing_tables_query = """ SELECT 
    'DROP VIEW IF EXISTS ' || table_schem || '.' || table_name || ';' AS table_drop_statement
FROM
    system.jdbc.tables
WHERE 
    True 
    AND table_type = 'VIEW'
    AND table_cat = 'enterprise_prod'
    AND table_schem <> 'information_schema'
    AND table_schem = 'staging'
"""
tables_to_drop_df=pd.read_sql(existing_tables_query, conn_starburst)

In [4]:
tables_to_drop_df.head()

,table_drop_statement
0,DROP VIEW IF EXISTS staging.stg_android__bills...
1,DROP VIEW IF EXISTS staging.stg_android__creat...
2,DROP VIEW IF EXISTS staging.stg_android__if_ho...
3,DROP VIEW IF EXISTS staging.stg_android__pay_b...
4,DROP VIEW IF EXISTS staging.stg_android__tracks;


In [5]:
' '.join(tables_to_drop_df['table_drop_statement'][290:].head())

'DROP VIEW IF EXISTS staging.stg_neo__se_cross_sell_get_approval_request_approve_presented; DROP VIEW IF EXISTS staging.stg_neo__se_cross_sell_get_approval_request_decline_fired; DROP VIEW IF EXISTS staging.stg_neo__se_cross_sell_get_approval_request_decline_presented; DROP VIEW IF EXISTS staging.stg_neo__se_cross_sell_get_approval_request_email_fired; DROP VIEW IF EXISTS staging.stg_neo__se_cross_sell_get_approval_request_email_presented;'

These statements can be run directly in the console (if available). In this case, the production console was mocked by passing command through `Bolt` schedule that could assume the production root user role.

---